In [7]:
import json
from pathlib import Path

import pandas as pd

RAW_DIR = Path("data/raw")

# Load raw users JSON saved by extract.py
with open(RAW_DIR / "users_raw.json", encoding="utf-8") as f:
    users_raw = json.load(f)

# Flatten nested fields (e.g. address -> city becomes address_city)
users_df = pd.json_normalize(users_raw, sep="_")

print("Shape:", users_df.shape)
print(users_df.columns.tolist())

Shape: (208, 52)
['id', 'firstName', 'lastName', 'maidenName', 'age', 'gender', 'email', 'phone', 'username', 'password', 'birthDate', 'image', 'bloodGroup', 'height', 'weight', 'eyeColor', 'ip', 'macAddress', 'university', 'ein', 'ssn', 'userAgent', 'role', 'hair_color', 'hair_type', 'address_address', 'address_city', 'address_state', 'address_stateCode', 'address_postalCode', 'address_coordinates_lat', 'address_coordinates_lng', 'address_country', 'bank_cardExpire', 'bank_cardNumber', 'bank_cardType', 'bank_currency', 'bank_iban', 'company_department', 'company_name', 'company_title', 'company_address_address', 'company_address_city', 'company_address_state', 'company_address_stateCode', 'company_address_postalCode', 'company_address_coordinates_lat', 'company_address_coordinates_lng', 'company_address_country', 'crypto_coin', 'crypto_wallet', 'crypto_network']


In [8]:
# Whitelist approach: keep only analytics-relevant columns and rename them to snake_case
USER_COLUMNS = {
    "id": "user_id",
    "firstName": "first_name",
    "lastName": "last_name",
    "age": "age",
    "gender": "gender",
    "role": "role",
    "address_city": "city",
    "address_state": "state",
    "address_stateCode": "state_code",
    "address_postalCode": "postal_code",
    "address_country": "country",
    "company_department": "company_department",
    "company_title": "job_title",
}

users_clean = users_df[list(USER_COLUMNS)].rename(columns=USER_COLUMNS)

print("Shape:", users_clean.shape)
users_clean.info()
users_clean.head()

Shape: (208, 13)
<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   user_id             208 non-null    int64
 1   first_name          208 non-null    str  
 2   last_name           208 non-null    str  
 3   age                 208 non-null    int64
 4   gender              208 non-null    str  
 5   role                208 non-null    str  
 6   city                208 non-null    str  
 7   state               208 non-null    str  
 8   state_code          208 non-null    str  
 9   postal_code         208 non-null    str  
 10  country             208 non-null    str  
 11  company_department  208 non-null    str  
 12  job_title           208 non-null    str  
dtypes: int64(2), str(11)
memory usage: 21.3 KB


,user_id,first_name,last_name,age,gender,role,city,state,state_code,postal_code,country,company_department,job_title
0,1,Emily,Johnson,29,female,admin,Phoenix,Mississippi,MS,29112,United States,Engineering,Sales Manager
1,2,Michael,Williams,36,male,admin,Houston,Alabama,AL,38807,United States,Support,Support Specialist
2,3,Sophia,Brown,43,female,admin,Washington,Alabama,AL,32822,United States,Research and Development,Accountant
3,4,James,Davis,46,male,admin,Seattle,Pennsylvania,PA,68354,United States,Support,Research Analyst
4,5,Emma,Miller,31,female,admin,Jacksonville,Colorado,CO,26593,United States,Human Resources,Quality Assurance Engineer


In [9]:
# Load raw products JSON saved by extract.py
with open(RAW_DIR / "products_raw.json", encoding="utf-8") as f:
    products_raw = json.load(f)

# Flatten nested dictionaries (dimensions, meta); list fields stay as-is
products_df = pd.json_normalize(products_raw, sep="_")

print("Shape:", products_df.shape)
print(products_df.columns.tolist())

# Show only columns that contain missing values
nulls = products_df.isnull().sum()
print("\nColumns with missing values:")
print(nulls[nulls > 0])

Shape: (194, 27)
['id', 'title', 'description', 'category', 'price', 'discountPercentage', 'rating', 'stock', 'tags', 'brand', 'sku', 'weight', 'warrantyInformation', 'shippingInformation', 'availabilityStatus', 'reviews', 'returnPolicy', 'minimumOrderQuantity', 'images', 'thumbnail', 'dimensions_width', 'dimensions_height', 'dimensions_depth', 'meta_createdAt', 'meta_updatedAt', 'meta_barcode', 'meta_qrCode']

Columns with missing values:
brand    92
dtype: int64


In [10]:
# Check which categories have missing brands (before filling)
print("Categories with missing brand:")
print(products_df[products_df["brand"].isnull()]["category"].value_counts())

# Whitelist approach: keep analytics-relevant columns and rename to snake_case
PRODUCT_COLUMNS = {
    "id": "product_id",
    "title": "product_name",
    "category": "category",
    "brand": "brand",
    "sku": "sku",
    "price": "price",
    "discountPercentage": "discount_percentage",
    "rating": "rating",
    "stock": "stock",
    "weight": "weight",
    "availabilityStatus": "availability_status",
    "minimumOrderQuantity": "minimum_order_quantity",
    "warrantyInformation": "warranty_information",
    "shippingInformation": "shipping_information",
    "returnPolicy": "return_policy",
    "meta_createdAt": "created_at",
}

products_clean = products_df[list(PRODUCT_COLUMNS)].rename(columns=PRODUCT_COLUMNS)

# Products without a brand are generic items, so label them explicitly
products_clean["brand"] = products_clean["brand"].fillna("Unbranded")

# Convert ISO timestamp text to a real datetime type
products_clean["created_at"] = pd.to_datetime(products_clean["created_at"])

print("\nShape:", products_clean.shape)
products_clean.info()

Categories with missing brand:
category
kitchen-accessories    30
groceries              27
sports-accessories     17
home-decoration         5
tops                    5
womens-dresses          5
womens-jewellery        3
Name: count, dtype: int64

Shape: (194, 16)
<class 'pandas.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   product_id              194 non-null    int64              
 1   product_name            194 non-null    str                
 2   category                194 non-null    str                
 3   brand                   194 non-null    str                
 4   sku                     194 non-null    str                
 5   price                   194 non-null    float64            
 6   discount_percentage     194 non-null    float64            
 7   rating                  194 non-null    float64   

In [11]:
# Explode the nested reviews list: one row per review, linked back to its product
reviews_df = pd.json_normalize(
    products_raw,
    record_path="reviews",   # the list to turn into rows
    meta=["id"],             # parent field to carry into each row
)

# Whitelist + snake_case (reviewerEmail dropped as PII)
REVIEW_COLUMNS = {
    "id": "product_id",
    "rating": "review_rating",
    "comment": "review_comment",
    "date": "review_date",
    "reviewerName": "reviewer_name",
}

reviews_clean = reviews_df[list(REVIEW_COLUMNS)].rename(columns=REVIEW_COLUMNS)
reviews_clean["review_date"] = pd.to_datetime(reviews_clean["review_date"])

# Surrogate key: a unique ID for each review (the source has none)
reviews_clean.insert(0, "review_id", range(1, len(reviews_clean) + 1))

print("Shape:", reviews_clean.shape)
print("Reviews per product (min/max):",
      reviews_clean.groupby("product_id").size().min(), "/",
      reviews_clean.groupby("product_id").size().max())
reviews_clean.head()

Shape: (582, 6)
Reviews per product (min/max): 3 / 3


,review_id,product_id,review_rating,review_comment,review_date,reviewer_name
0,1,1,3,Would not recommend!,2025-04-30 09:41:02.053000+00:00,Eleanor Collins
1,2,1,4,Very satisfied!,2025-04-30 09:41:02.053000+00:00,Lucas Gordon
2,3,1,5,Highly impressed!,2025-04-30 09:41:02.053000+00:00,Eleanor Collins
3,4,2,5,Great product!,2025-04-30 09:41:02.053000+00:00,Savannah Gomez
4,5,2,4,Awesome product!,2025-04-30 09:41:02.053000+00:00,Christian Perez


In [12]:
# Load raw carts JSON saved by extract.py
with open(RAW_DIR / "carts_raw.json", encoding="utf-8") as f:
    carts_raw = json.load(f)

print("Number of carts:", len(carts_raw))

# Cart-level fields (one value per cart)
print("\nCart-level keys:", list(carts_raw[0].keys()))

# Product-level fields (one value per product inside a cart)
print("Product-level keys:", list(carts_raw[0]["products"][0].keys()))

# How many products each cart holds -> total tells us the expected row count after exploding
products_per_cart = pd.Series([len(cart["products"]) for cart in carts_raw])
print("\nProducts per cart (min/max):", products_per_cart.min(), "/", products_per_cart.max())
print("Expected rows after explode:", products_per_cart.sum())

Number of carts: 208

Cart-level keys: ['id', 'products', 'total', 'discountedTotal', 'userId', 'totalProducts', 'totalQuantity']
Product-level keys: ['id', 'title', 'price', 'quantity', 'total', 'discountPercentage', 'discountedTotal', 'thumbnail']

Products per cart (min/max): 2 / 6
Expected rows after explode: 800


In [13]:
# Explode the nested products list: one row per product per cart
cart_items_df = pd.json_normalize(
    carts_raw,
    record_path="products",        # the list to turn into rows
    meta=["id", "userId"],         # cart-level fields to carry into each row
    meta_prefix="cart_",           # avoids clash: cart "id" vs product "id"
)

# Whitelist + snake_case (title dropped: already in products_clean; thumbnail not needed)
CART_ITEM_COLUMNS = {
    "cart_id": "cart_id",
    "cart_userId": "user_id",
    "id": "product_id",
    "price": "price",
    "quantity": "quantity",
    "total": "total",
    "discountPercentage": "discount_percentage",
    "discountedTotal": "discounted_total",
}

cart_items_clean = cart_items_df[list(CART_ITEM_COLUMNS)].rename(columns=CART_ITEM_COLUMNS)

# Validation checks
print("Shape:", cart_items_clean.shape)
print("Nulls:", cart_items_clean.isnull().sum().sum())
print("Duplicate (cart_id, product_id):", cart_items_clean.duplicated(["cart_id", "product_id"]).sum())

# Referential integrity: every product/user must exist in its clean dimension table
print("Unknown product_ids:", (~cart_items_clean["product_id"].isin(products_clean["product_id"])).sum())
print("Unknown user_ids:", (~cart_items_clean["user_id"].isin(users_clean["user_id"])).sum())

cart_items_clean.info()
cart_items_clean.head()

Shape: (800, 8)
Nulls: 0
Duplicate (cart_id, product_id): 12
Unknown product_ids: 0
Unknown user_ids: 0
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cart_id              800 non-null    object 
 1   user_id              800 non-null    object 
 2   product_id           800 non-null    int64  
 3   price                800 non-null    float64
 4   quantity             800 non-null    int64  
 5   total                800 non-null    float64
 6   discount_percentage  800 non-null    float64
 7   discounted_total     800 non-null    float64
dtypes: float64(4), int64(2), object(2)
memory usage: 50.1+ KB


,cart_id,user_id,product_id,price,quantity,total,discount_percentage,discounted_total
0,1,1,162,29.99,4,119.96,12.13,105.41
1,1,1,113,3999.99,3,11999.97,12.10,10547.97
2,1,1,122,299.99,3,899.97,6.69,839.76
3,1,1,138,8.99,2,17.98,1.71,17.67
4,2,2,86,19.99,5,99.95,6.83,93.12


In [14]:
# Cart header table: one row per cart (products list excluded, already exploded)
CART_COLUMNS = {
    "id": "cart_id",
    "userId": "user_id",
    "total": "total",
    "discountedTotal": "discounted_total",
    "totalProducts": "total_products",
    "totalQuantity": "total_quantity",
}

carts_df = pd.json_normalize(carts_raw)
carts_clean = carts_df[list(CART_COLUMNS)].rename(columns=CART_COLUMNS)

# Meta columns from json_normalize can arrive as generic "object" type; align ID types for joining
cart_items_clean[["cart_id", "user_id"]] = cart_items_clean[["cart_id", "user_id"]].astype("int64")

# Recompute cart totals from the line items
items_agg = cart_items_clean.groupby("cart_id").agg(
    items_total=("total", "sum"),
    items_discounted_total=("discounted_total", "sum"),
    items_count=("product_id", "count"),
    items_quantity=("quantity", "sum"),
).reset_index()

recon = carts_clean.merge(items_agg, on="cart_id", how="left")

# Reconciliation: header values must equal the sum of their line items (0.01 tolerance for float rounding)
print("Shape:", carts_clean.shape)
print("Nulls:", carts_clean.isnull().sum().sum())
print("Total mismatches:", ((recon["total"] - recon["items_total"]).abs() > 0.01).sum())
print("Discounted total mismatches:", ((recon["discounted_total"] - recon["items_discounted_total"]).abs() > 0.01).sum())
print("Product count mismatches:", (recon["total_products"] != recon["items_count"]).sum())
print("Quantity mismatches:", (recon["total_quantity"] != recon["items_quantity"]).sum())

carts_clean.head()

Shape: (208, 6)
Nulls: 0
Total mismatches: 0
Discounted total mismatches: 0
Product count mismatches: 0
Quantity mismatches: 0


,cart_id,user_id,total,discounted_total,total_products,total_quantity
0,1,1,13037.88,11510.81,4,12
1,2,2,139.93,125.70,2,7
2,3,3,1794.85,1625.77,6,15
3,4,4,689.93,636.81,2,7
4,5,5,1467.88,1205.80,3,12
